# Basic usage example for simulating PEN DNA systems

You will need two elements from the Python side:
- A representation of your system (DACCADIndividual)
- A wrapper for the DACCAD simulator

In this example, we will simulate a basic Oligator (chemical oscillator) with the Jpype wrapper.

In [ ]:
from daccadevo.core import DaccadIndividual, evaluate_timeseries
from daccadevo.wrappers import Jpype_wrapper
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
indiv = DaccadIndividual([0,1])
indiv.resize(3) # We need two signal species
indiv.stabilities = np.array([74.328, 74.328, 74.328]) # represent the stabilities of signal sequences on their templates
indiv.activations = np.array([[10.0,10.0,0.0],[0.0,0.0,10.0],[0.0,0.0,0.0]]) # 0 -> 0, 0 -> 1 activations of signals
indiv.inhibitions = np.zeros((3,3,3))
indiv.inhibitions[2,0,0] = 10.0 # 2 -> Inh -| (0 -> 0) 1 triggers the production of the inhibitor of 0 -> 0
indiv.assemble() # compile all informations into a single array representation

In [ ]:
# Basic wrapper and DACCAD configurations
# Not specifying anything would use defaults
config = {'debug': False, 
          'dataDir': '.', 'daccad': {'config_file':'./short.conf' ,
                                     'executable_path':'../../daccad'}
         }

In [ ]:
wrapper = Jpype_wrapper(config=config)

In [ ]:
initConc = [0.0 for _ in range(indiv.nb_nodes)]
initConc[0] = 0.1 # initial conditions

In [ ]:
results = evaluate_timeseries(indiv,config=config, initConc=initConc,wrapper=wrapper, scales=[1.0,1.0])

In [ ]:
plt.plot(results[:,:indiv.nb_nodes+1]) # Plot the inhibitor as well
ax = plt.gca()
ax.legend([str(i) for i in range(indiv.nb_nodes+1)],loc=(1.01, 0.01))
plt.tight_layout()

# Optimization against a custom target

When performing the optimization without the daccadevo executable, you need to:

- define (or, in the present case, import) the evaluation function
- create an optimizer
- run the optimization
- extract the best result, plot its behavior and extract some statistics on it

In [ ]:
# First, import the evaluation function and 
# the convenience class for optimization
from bistable_eval import bistable_eval_fn
from daccadevo.optimization.experiment import DACCADExperiment

In [ ]:
# Setup the optimizer using the configuration file bistable.yaml
# For an actual optimization run, do not use seeds.
exp = DACCADExperiment("bistable.yaml", seed=1234)

In [ ]:
exp.run()

In [ ]:
# Results should be in `results/bistable/final_[date].p`
# We will use glob to open the last available file
from glob import glob
import pickle

In [ ]:
with open(sorted(glob("results/bistable/final*.p"))[-1],"rb") as file:
    res = pickle.load(file)

In [ ]:
# The results are stored in the QDPy format
# Get the best individual
res["container"].best

In [ ]:
# Simulate both initial conditions, showing bistability
high_conc = 10
low_conc=0.1 
initConc1 = ["0.0" for _ in range(res["container"].best.nb_nodes)]
initConc2 = ["0.0" for _ in range(res["container"].best.nb_nodes)]
initConc1[0] = high_conc
initConc2[1] = high_conc
initConc1[1] = low_conc
initConc2[0] = low_conc
y1 = evaluate_timeseries(res["container"].best, config = config, initConc=initConc1)[:,:2]
y2 = evaluate_timeseries(res["container"].best, config = config, initConc=initConc2)[:,:2]

In [ ]:
# Blue high, orange low
plt.plot(y1)

In [ ]:
# Blue low orange high
plt.plot(y2)

In [ ]:
# Details of statistics on the best individual
res["container"].best.scores

# Use visualization tools to check the shape of an individual

In [ ]:
from PySide6.QtWidgets import QApplication
from daccadevo.visualization.window import MainWindow

In [ ]:
network_sequence = res["container"].best
dict_network = {}
dict_network['stabilities'] = getattr(network_sequence, 'stabilities', None)
dict_network['activations'] = getattr(network_sequence, 'activations', None)
dict_network['inhibitions'] = getattr(network_sequence, 'inhibitions', None)
dict_network['pseudo_templates'] = getattr(network_sequence, 'pseudo_templates', None)
dict_network['predator_prey_templates'] = getattr(network_sequence, 'predator_prey_templates', None)

In [ ]:
%gui qt
app = QApplication()
window = MainWindow(dict_network=dict_network, key="best", show_inhibitors = True)
window.show()
# Note: opens in a separate window